# Gerar pela API ou conferir uma proposta

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="02_teste_inferencia_planilha_turmas"


{'mode': 'GENERATE_AND_VALIDATE',
 'proposal_id': None,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'model_name': 'hi.demo.notebook.a9291c01',
 'model_version': '1',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': True,
 'confirmation': 'EXECUTE_DEMO',
 'request_key': 'proposal-a9291c01',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 3,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': True,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Preflight e seleção. READ_EXISTING não escreve; GENERATE_AND_VALIDATE envia POST ao worker. Escolha explicitamente snapshot e modelo/versão; confirme a escrita e defina request_key.

In [2]:
from src.workbench.contract_review import preflight,candidates,fetch_payload,reconcile
display(pd.DataFrame(preflight(cfg)));display(pd.DataFrame(candidates(cfg)))
proposal_id=str(params.proposal_id) if params.proposal_id else ""


,dependency,status
0,API/OpenAPI/auth negative/MLflow SDK,PASS


,execution_id,generated_at,model_name,model_version,status,row_count
0,8813ad68-5431-46c8-b8d6-2b4e9d3e4f6c,2026-11-04 21:42:57.423889+00:00,planilha-turmas-demo,2,DRAFT,90
1,2d45731a-1625-4b5c-8ac0-4986e996cdf6,2026-11-04 21:42:50.973905+00:00,hi.demo.v3-negative.699bb4f2,1,DRAFT,90
2,bf3e3b0d-3c95-4065-8442-5934dbe307da,2026-11-04 21:41:49.942463+00:00,hi.demo.integration.70491a51,1,DRAFT,90
3,b7ad5311-3fe1-4f7f-9956-beec9696d0fd,2026-11-04 21:41:41.732445+00:00,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,2,DRAFT,9
4,37ce4733-c129-4404-997d-a9f630d81da9,2026-11-04 21:26:07.344647+00:00,hi.demo.probe.18cb82de,1,DRAFT,90
5,6da4d8cc-c00b-4607-8e18-9da143d2f814,2026-11-04 21:25:58.888545+00:00,planilha-turmas-demo,1,DRAFT,90


2. Submeter geração somente no modo opt-in. Nenhuma persistência é feita pelo notebook. Esperado: job, correlation_id e status_url.

In [3]:
if params.mode=="GENERATE_AND_VALIDATE":
    params.authorize()
    if not params.snapshot_id or not params.model_version:
        raise ValueError("Selecione snapshot_id e model_version explicitamente")
    from src.workbench.operations_sdk import generate
    payload=dict(cycle_code=cfg.cycle,snapshot_id=str(params.snapshot_id),
        model_selectors=[dict(model_name=params.model_name,model_version=params.model_version)],
        capacity=params.capacity,requested_by=params.requester,confirm_demo=True)
    display(payload)
    submitted=generate(cfg,payload,params.request_key)
    display(submitted)
    finished=await_job(cfg,submitted["status_url"])
    if finished["status"]!="COMPLETED":
        raise RuntimeError("Geração não completou; consulte o job persistido: "+str(finished))
    proposal_id=finished["proposal_id"]
    evidence.update(request=payload,job=finished)
if not proposal_id:
    raise ValueError("Selecione proposal_id da tabela ou configure modo GENERATE_AND_VALIDATE")


{'cycle_code': '2026-1',
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'model_selectors': [{'model_name': 'hi.demo.notebook.a9291c01',
   'model_version': '1'}],
 'capacity': 40,
 'requested_by': 'scientist-demo',
 'confirm_demo': True}

{'generation_job_id': '605bb885-2040-43da-934e-752305cc6215',
 'job_id': '605bb885-2040-43da-934e-752305cc6215',
 'correlation_id': 'c268fc9b-98c2-4f73-9e00-52fea1f6ec89',
 'status': 'PENDING',
 'attempt': 0,
 'error_code': None,
 'status_url': '/v1/proposals/generations/605bb885-2040-43da-934e-752305cc6215',
 'proposal_id': None,
 'proposal_url': None,
 'trace_id': None,
 'started_at_utc': None,
 'ended_at_utc': None}

{'status': 'PENDING', 'counts': None, 'utc': '2026-11-04T21:43:15.597733+00:00'}


{'status': 'COMPLETED', 'counts': None, 'utc': '2026-11-04T21:43:17.609370+00:00'}


3. Obter payload integral paginado e validar schema/hash. Este é o JSON consumido pela aplicação.

In [4]:
pages,offers=fetch_payload(cfg,proposal_id,page_size=50)
for page in pages: display(page)
display(pd.json_normalize(offers))
display(pd.DataFrame(pages[0]["validation"]))
assert not any(r["status"]=="FAIL" for r in pages[0]["validation"])
evidence.update(proposal_id=proposal_id,http_pages=pages)


{'schema_version': 'poc-planilha-turmas-v1',
 'contract_status': 'PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO',
 'proposal_id': '605bb885-2040-43da-934e-752305cc6215',
 'execution_id': '605bb885-2040-43da-934e-752305cc6215',
 'proposal_revision': 1,
 'academic_cycle': {'academic_cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
  'cycle_code': '2026-1'},
 'status': 'DRAFT',
 'source_type': 'DEMO',
 'generated_at_utc': '2026-11-04T21:43:15.887888Z',
 'lineage': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
  'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
  'training_campaign_id': None,
  'ruleset_version': 'POC-whole-cohort-ceil-capacity-v1',
  'model_usages': [{'usage_id': '7d57e279-15d0-4c1c-a254-c5c735afc4fd',
    'proposal_id': '605bb885-2040-43da-934e-752305cc6215',
    'inference_id': '2af2e452-b78c-4a86-a735-92f3276f9baa',
    'model_key': 'hi.demo.n

{'schema_version': 'poc-planilha-turmas-v1',
 'contract_status': 'PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO',
 'proposal_id': '605bb885-2040-43da-934e-752305cc6215',
 'execution_id': '605bb885-2040-43da-934e-752305cc6215',
 'proposal_revision': 1,
 'academic_cycle': {'academic_cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
  'cycle_code': '2026-1'},
 'status': 'DRAFT',
 'source_type': 'DEMO',
 'generated_at_utc': '2026-11-04T21:43:15.887888Z',
 'lineage': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
  'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
  'training_campaign_id': None,
  'ruleset_version': 'POC-whole-cohort-ceil-capacity-v1',
  'model_usages': [{'usage_id': '7d57e279-15d0-4c1c-a254-c5c735afc4fd',
    'proposal_id': '605bb885-2040-43da-934e-752305cc6215',
    'inference_id': '2af2e452-b78c-4a86-a735-92f3276f9baa',
    'model_key': 'hi.demo.n

,offer_id,campus_id,course_id,period_id,discipline_id,cohort,shift,offer_type,status,forecast_raw,...,students.regular,students.dependency,students.elective,students.total,students.origins,modulation.rule_version,modulation.capacity,modulation.proposed_groups,modulation.groups,modulation.provisional
0,c1c5b95d-bf48-59e4-9003-0a5dd8d9232b,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,None,None,None,SUGERIDA,102.089955,...,None,None,None,103,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
1,ece32b05-cec4-55d8-aa95-abfa3e196231,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,None,None,None,SUGERIDA,102.089955,...,None,None,None,103,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
2,32ac953b-f44c-54f3-9eb8-9c2ef4774a53,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,None,None,None,SUGERIDA,102.089955,...,None,None,None,103,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
3,65e47132-3e4a-50e2-9294-c0a4b354dd7c,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,None,None,None,SUGERIDA,103.245511,...,None,None,None,104,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
4,5a70e168-7a30-5029-91ed-59160fd81c76,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,None,None,None,SUGERIDA,103.245511,...,None,None,None,104,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,4e9fb201-c5f2-5ad8-89a6-6e3e6c065e1b,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,11e42499-2538-5286-a8b9-3fda55802247,d30d0d0a-6af3-5ba7-b60d-e25d687c1e69,None,None,None,SUGERIDA,96.586051,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
86,946dc347-7c59-5618-ae70-1c626bdbb7f0,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,11e42499-2538-5286-a8b9-3fda55802247,e7f05bc1-407f-5670-84d7-aae409217d6f,None,None,None,SUGERIDA,96.586051,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
87,de6d5112-28c1-5b46-bf1a-f41d2023e5dd,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,1494b22f-3999-5714-ba12-3e8a1c65117c,5ebf59a4-4457-529e-98cb-04d804233a7c,None,None,None,SUGERIDA,96.497162,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
88,43dc2a6b-63d3-56f0-b660-f79f35f6599b,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,1494b22f-3999-5714-ba12-3e8a1c65117c,7064e2c5-ce43-55f0-b151-5d1dd109de0f,None,None,None,SUGERIDA,96.497162,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True


,rule_id,status,reason,evidence
0,POC-DRAFT,PASS,Proposta demonstrativa sem aprovação automática,proposal_generations.status
1,RF008-partial,PASS,Sem docente/horário/sala na representação inic...,offers.teacher_suggestion
2,RGN001/US027,NOT_EVALUATED,Decisão humana e prevalência de edições não ex...,DEMO scope
3,RGN002/RF010,NOT_EVALUATED,Integração institucional entre todos os cursos...,DEMO scope
4,RGN003/RF021–025/US029–030,NOT_EVALUATED,Alvo agregado; composição regular/dependência/...,DEMO scope
5,RGN023/US028,NOT_EVALUATED,Gate de validação da matriz depende da aplicação,DEMO scope
6,RF009/RF020/US026,NOT_EVALUATED,"Turno, coorte de ingresso e matriz homologada ...",DEMO scope
7,RF040–047/US033–034,NOT_EVALUATED,Capacidade DEMO não é regra institucional; uni...,DEMO scope
8,RF117–125,NOT_EVALUATED,Sem elegibilidade PRIME/Escola/turno ou aprova...,DEMO scope
9,RF155–159/US005–006,NOT_EVALUATED,SQL sintético e temporal; nenhuma integração i...,DEMO scope


4. Reconciliação somente leitura: linhas SQL, versões Registry, snapshot e bytes RustFS. Esperado: mesmos IDs, quantidades e hashes.

In [5]:
evidence["reconciliation"]=reconcile(cfg,proposal_id,pages,offers)
display(evidence["reconciliation"])
display(Markdown(f"[API](http://localhost:8000/v1/proposals/{proposal_id}/application-view) · [MLflow]({cfg.ui}) · [RustFS](http://localhost:9001)"))


{'sql': {'database': 'planejamento_db',
  'schema': 'planning',
  'status': 'PASS',
  'execution_id': '605bb885-2040-43da-934e-752305cc6215',
  'run_id': '420b1e103c8741ddbb8ec66427a95c0b',
  'model_version': '1',
  'proposal_status': 'DRAFT',
  'stored_rows': 90,
  'proposed_students': 8694,
  'proposed_groups': 270,
  'query': 'sql/queries/verify_proposal.sql'},
 'storage': [{'bucket': 'pucpr-poc',
   'key': 'mlflow/3/models/m-053c00350db44eb1a28c5dae0ed54a66/artifacts/MLmodel',
   'bytes': 1336,
   'sha256': 'd1ba43508ec6fc0a1d34d487a39af29294c7af2212affc3f8bd263118f5d6c79',
   'object_version_id': '203e4bec-7aef-40ba-92c5-bf1182776cf7',
   'model_version': '1',
   'status': 'PASS'},
  {'bucket': 'pucpr-poc',
   'key': 'mlflow/3/models/m-053c00350db44eb1a28c5dae0ed54a66/artifacts/model.pkl',
   'bytes': 8671,
   'sha256': '4268666504b258b661b9c6cc009bdf8c21d06fde4317fb389a585e633be3ad0b',
   'object_version_id': '8ef84859-a09a-4855-8bfc-85178ba7212f',
   'model_version': '1',
   'st

[API](http://localhost:8000/v1/proposals/605bb885-2040-43da-934e-752305cc6215/application-view) · [MLflow](http://127.0.0.1:5000) · [RustFS](http://localhost:9001)

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [6]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v3/02_teste_inferencia_planilha_turmas.json'